![Astrofisica Computacional](../../new_logo.png)

# Computational Astrophysics –  Multilinear Fit with SciKit-Learn

## Dr. rer. nat. Jose Ivan Campos Rozo<sup>1,2</sup>

1. Astronomical Institute of the Czech Academy of Sciences\
   Department of Solar Physics\
   Ondřejov, Czec Republic

2. Observatorio Astronómico Nacional\
   Facultad de Ciencias\
   Universidad Nacional de Colombia

e-mail: jicamposr@unal.edu.co & rozo@asu.cas.cz)

---
Taken from previous lectures of this course.


### About this notebook

In this worksheet, we will use the `scikit-learn` package split a sample database of 88 supermassive black holes into train and test sets in order to implement a multidimensional simple linear regression, a Ridge regression and a Lasso regression.

---

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

import pandas as pd
from astropy.io import ascii

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# No warnings
import warnings
warnings.filterwarnings('ignore')

### Green and Ho data

Consider the dataset reported by Greene and Ho (2006), containing the features of 88 galaxies. 

Greene, J. E. and Ho, L. C. *The MBH − σ∗ Relation in Local Active Galaxies*. ApJ 641 L21 (2006)
https://ui.adsabs.harvard.edu/abs/2006ApJ...641L..21G/abstract

The dataset is available online in various formats at

http://vizier.cfa.harvard.edu/viz-bin/VizieR?-source=J/ApJ/641/L21.


In [ ]:
data = ascii.read('SMBHData/table1.dat', readme='SMBHData/ReadMe')
data

Now, we convert the astropy Table to a pandas dataframe using the `.to_pandas()` method. Details at

https://docs.astropy.org/en/stable/api/astropy.table.Table.html#astropy.table.Table.to_pandas

In [ ]:
df = data.to_pandas()
df.describe()

The dataframe includes data from 88 supermassive black holes. The columns correspond to

**z** : Redshift \
**sigma**\* : Stellar velocity dispersion \
**e_sigma**\* : Formal uncertainty in sigma* \
**FWHM** : H<sub>$\alpha$</sub> Full-Width at Half Maximum \
**e_FWHM** : Formal uncertainty in FWHM \
**logL** : $\log_{10}$ of H<sub>$\alpha$</sub> luminosity in erg/s \
**e_logL** : Formal uncertainty in logL \
**logM** : $\log_{10}$ of the Black Hole mass \
**E_logM** : Formal (upper limit) uncertainty in logM \
**e_logM** : Formal (lower limit) uncertainty in logM 


We will make a linear fit between the variables $\log M$ and $\log \left( \frac{\sigma_*}{\sigma_0} \right)$ and between the varibles $\log M$ and $\log  \text{FWHM} $, where $\sigma_0 = 200.$ is a reference value given by the authors. Hence, we add the corresponding columns to the dataframe.

In [ ]:
sigma0 = 200.
df['logsigma*'] = np.log10(df['sigma*']/sigma0)
df['logFWHM'] = np.log10(df['FWHM'])
df.describe()

### Multidimensional Linear Fit using  `scikit-learn`

From the `scikit-learn.linear_model` package, we will import the function `LinearRegression`. Complete documentation about it can be found at

https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html

This function fits a linear model to minimize the residual sum of squares between the observed targets in the dataset and the targets predicted by the linear approximation (ordinary least squares fit).

Before making the fit, we need to prepare our data. First, we handle the missing values in the features,

In [ ]:
newdf = df[['logM','logsigma*','logFWHM']]
newdf = newdf.apply (pd.to_numeric, errors='coerce')
newdf = newdf.dropna()
newdf.describe()

Now, we define two new dataframes with the important features for the linear model. We use the capital letter X to identify the independent parameter data because it will be an array for the multidimensional fit.

In [ ]:
Xdf = newdf[['logsigma*','logFWHM']]
ydf = newdf[['logM']]

We have 71 samples with three features. 

In [ ]:
Xdf.describe() 

In [ ]:
ydf.describe()

We will separate the dataframes into two sets, a training set and a testing set. We will use the function `train_test_split` function from the `sklearn.model_selection` package. We will use a size of 10% for the test set. More information on this function at

https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(Xdf, ydf, random_state=60, test_size=0.15)

We obtained the following sets 

In [ ]:
X_train.shape , y_train.shape

In [ ]:
X_test.shape , y_test.shape

Now, we train the linear model

$y = \theta_0 + \theta_1 x_1 + \theta_2 x_2$ ,

with the method `LinearRegression().fit()` and the training data

In [ ]:
lr = LinearRegression().fit(X_train, y_train)

The parameters obtained from the lienar fit are recovered with the methods `.intercept_` ($\theta_0$) and `.coef_` ($\theta_1$ and $\theta_2$).

In [ ]:
lr.intercept_ , lr.coef_ 

Using the method `.score()`, which returns the coefficient of determination $R^2$ for the model, we obtain, for the training and the testing sets, the following results

In [ ]:
lr.score(X_train, y_train)

In [ ]:
lr.score(X_test, y_test)

In [ ]:
# Plot
predictions = lr.predict(X_test)
#predictions2 = lr.predict(X_train)

plt.figure(figsize=(10,7))
#plt.scatter(y_train, predictions2, alpha=0.5, color='violet', label='Train subset')
plt.scatter(y_test, predictions, alpha=0.5, label='Test subset')
plt.plot(y_train, y_train, '--', color='crimson')
plt.ylabel(r'predictions')
plt.xlabel(r'targets')
plt.legend()
plt.show()

#### Score of the linear model depending on the train set size


In [ ]:
training_acc = []
test_acc = []

training_size = np.linspace(0.1,0.9,50)

for s in training_size:
    X_train, X_test, y_train, y_test = train_test_split(Xdf, ydf, random_state=60, train_size=s)
    lr = LinearRegression().fit(X_train, y_train)
    training_acc.append(lr.score(X_train,y_train))
    test_acc.append(lr.score(X_test,y_test))
    
plt.plot(training_size, training_acc, label='train accuracy')
plt.plot(training_size, test_acc, label='test accuracy')
plt.xlabel(r'Train size')
plt.ylabel(r'Accuracy')
plt.legend()
plt.show()

---
---
### Ridge Regression
From the `scikit-learn.linear_model` package, we will import the function `Ridge`. Complete documentation about it can be found at

https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html

This function fits a linear model to minimize the residual sum of squares between the observed targets in the dataset and the targets predicted by the linear approximation (ordinary least squares fit), but it is regularized with the L2-norm.

**This is a method of regularization of ill-posed problems and it is particularly useful to mitigate the problem of multicollinearity in linear regression, which commonly occurs in models with large numbers of parameters.**

In [ ]:
from sklearn.linear_model import Ridge

X_train, X_test, y_train, y_test = train_test_split(Xdf, ydf, random_state=50, test_size=0.1)
ridge = Ridge().fit(X_train, y_train)

ridge.intercept_ , ridge.coef_

In [ ]:
ridge.score(X_train, y_train)

In [ ]:
ridge.score(X_test, y_test)

#### Score of the linear model depending on the train set size

In [ ]:
training_acc = []
test_acc = []

training_size = np.linspace(0.1,0.9,50)

for s in training_size:
    X_train, X_test, y_train, y_test = train_test_split(Xdf, ydf, random_state=50, train_size=s)
    ridge = Ridge().fit(X_train, y_train)
    training_acc.append(ridge.score(X_train,y_train))
    test_acc.append(ridge.score(X_test,y_test))
    
plt.plot(training_size, training_acc, label='train accuracy')
plt.plot(training_size, test_acc, label='test accuracy')
plt.xlabel(r'Train size')
plt.ylabel(r'Accuracy')
plt.legend()
plt.show()

---
### LASSO Regression

From the `scikit-learn.linear_model` package, we will import the function `LASSO`. Complete documentation about it can be found at

https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html

This function fits a linear model to minimize the residual sum of squares between the observed targets in the dataset and the targets predicted by the linear approximation (ordinary least squares fit), but it is regularized with the L1-norm.

The **L**east **A**bsolute **S**hrinkage and **S**election **O**perator (**LASSO**) is a regression analysis method that performs both variable selection and regularization in order to enhance the prediction accuracy and interpretability of the resulting statistical model.

In [ ]:
from sklearn.linear_model import Lasso
X_train, X_test, y_train, y_test = train_test_split(Xdf, ydf, random_state=50, test_size=0.1)
lasso = Lasso(alpha=0.1).fit(X_train, y_train)

lasso.intercept_ , lasso.coef_

In [ ]:
lasso.score(X_train, y_train)

In [ ]:
lasso.score(X_test, y_test)

#### Score of the linear model depending on the train set size


In [ ]:
training_acc = []
test_acc = []

training_size = np.linspace(0.1,0.9,50)

for s in training_size:
    X_train, X_test, y_train, y_test = train_test_split(Xdf, ydf, random_state=50, train_size=s)
    lasso = Lasso(alpha=0.1).fit(X_train, y_train)
    training_acc.append(lasso.score(X_train,y_train))
    test_acc.append(lasso.score(X_test,y_test))
    
plt.plot(training_size, training_acc, label='train accuracy')
plt.plot(training_size, test_acc, label='test accuracy')
plt.xlabel(r'Train size')
plt.ylabel(r'Accuracy')
plt.legend()
plt.show()